In [ ]:
from bs4 import BeautifulSoup
import requests
from typing import Dict
from typing import List
import csv
from time import sleep

In [ ]:
URL = 'https://www.korpuspjm.uw.edu.pl/videos/'
FIELDS = ['video_id', 'video_url', 'subtitles_url', 'video_type', 'age', 'sex', 'task_id', 'parse_status']
def parse_page(id: int) -> Dict[str, str]:
  data = {k: None for k in FIELDS}
  data['video_id'] = f'{id}'

  resp = requests.get(f'{URL}{id}')

  if resp.status_code != 200:
    data['parse_status'] = f'status code: {resp.status_code}'
    resp.close()
    return data

  soup = BeautifulSoup(resp.text, 'html.parser')
  resp.close()

  video_tag = soup.find('source', type='video/mp4')
  subtitles_tag = soup.find('track', kind='subtitles')
  data['video_url'] = video_tag.get('src')
  data['subtitles_url'] = subtitles_tag.get('src')

  video_type_tag = soup.find('ul', class_='breadcrumb')
  data['video_type'] = ';'.join([tag.text for tag in video_type_tag.contents])

  metadata_tag = soup.find('div', class_='event-details-panel')
  metadata_tag = metadata_tag.find_all('p')
  for paragraph in metadata_tag:
    key = None
    match paragraph.contents[0].text:
      case 'Płeć':
        key = 'sex'
      case 'Wiek':
        key = 'age'
      case 'Numer zadania':
        key = 'task_id'
    if key:
      data[key] = paragraph.contents[1].text

  data['parse_status'] = 'ok'

  return data

In [ ]:
def parse_pages(ids: List[int], file_path: str):
  with open(file_path, 'w') as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for id in ids:
      data = parse_page(id)
      writer.writerow(data.values())
      sleep(0.1)

In [ ]:
parse_pages(range(1001, 2000), 'beyond_1000.csv')

KeyboardInterrupt: 

### Explore downloaded

In [ ]:
# pjm_open_parsed.csv is the CSV written by parse_pages() above (the scrape of the corpus pages).
# The copy used for the thesis was kept on a private Drive and is not public; rerun the scrape to regenerate it.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('pjm_open_parsed.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   video_id       1000 non-null   int64  
 1   video_url      532 non-null    object 
 2   subtitles_url  532 non-null    object 
 3   video_type     532 non-null    object 
 4   age            532 non-null    object 
 5   sex            532 non-null    object 
 6   task_id        532 non-null    float64
 7   parse_status   1000 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 62.6+ KB


In [ ]:
df = df[df['parse_status'] == 'ok']

In [ ]:
df.task_id = df.task_id.astype(int)

In [ ]:
df.task_id.value_counts()

,count
task_id,
17,142
8,128
15,71
13,67
24,64
7,60


In [ ]:
df.video_type.unique()

array(['monolog;obraz;komiks;Frog Story',
       'monolog;film;animacja;Canary Row',
       'dialog;obraz;tekst;Spotkajmy się',
       'monolog;film;film w PJM;Alarm pożarowy',
       'monolog;film;film aktorski;Brzdąc',
       'monolog;film;film aktorski;Pear Stories',
       'dialog;obraz;ilustracja;Co to za znak?',
       'dialog;obraz;komiks;Opowiedz mi komiks'], dtype=object)

In [ ]:
df[df['video_type'].str.find('dialog') != -1].count()

,0
video_id,198
video_url,198
subtitles_url,198
video_type,198
age,198
sex,198
task_id,198
parse_status,198


In [ ]:
df[df['video_type'].str.find('monolog') != -1].count()

,0
video_id,334
video_url,334
subtitles_url,334
video_type,334
age,334
sex,334
task_id,334
parse_status,334


In [ ]:
df.sex.value_counts()

,count
sex,
F,178
M,156
FF,85
MM,70
MF,43


In [ ]:
df.age.value_counts()

,count
age,
32-45,125
46-60,85
18-30,65
60+,59
"32-45, 32-45",50
"18-30, 18-30",33
"46-60, 46-60",26
"60+, 60+",19
"46-60, 32-45",18


In [ ]:
df = df.drop(['parse_status'], axis=1)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 532 entries, 1 to 736
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       532 non-null    int64 
 1   video_url      532 non-null    object
 2   subtitles_url  532 non-null    object
 3   video_type     532 non-null    object
 4   age            532 non-null    object
 5   sex            532 non-null    object
 6   task_id        532 non-null    int64 
dtypes: int64(2), object(5)
memory usage: 49.4+ KB


In [ ]:
df.to_csv('clean_pjm_open_parsed.csv', index=False)